In [3]:
from simulation import citygraph_dataset
# from learning import inductive_route_learning, eval_route_generator, bee_colony
from learning.bee_colony import main as main_bee  # я так обозвал
from learning.eval_route_generator import main as main_eval # я так обозвал
from learning.nsgaii import main as main_ngsaii
# from learning.simulated_annealing import main as main_sa
# from learning.bagloee import main as main_bagloo
from omegaconf import OmegaConf, DictConfig
from simulation import drawing

from tqdm import tqdm
from pathlib import Path

In [4]:
from hydra import initialize_config_dir, compose
from omegaconf import OmegaConf
import os

cfg_dir = os.path.abspath("../TNDP_learning/cfg")

### NSGAII

In [5]:
dataset_name = 'mandl'
demand_time_weight = 0.33
route_time_weight = 0.33
median_connectivity_weight = 0.33
experiment_name = f'exp_weighted_connectivity_{dataset_name}_pp_{demand_time_weight}_op_{route_time_weight}_cp_{median_connectivity_weight}'
initial_routes_name = experiment_name+ '_starting'
generated_routes_name = experiment_name + '_generated'

In [8]:
with initialize_config_dir(config_dir=cfg_dir, version_base=None):
    
    cfg_eval = compose(
        config_name="nsgaii",   # из @hydra.main
        overrides=[
            f"+eval={dataset_name}", # конфиг в котором задается кол-во маршрутов и их макс и мин длины (можно заменить на vo для теста по ваське)
            "+model_1.weights=../TNDP_learning/output/inductive_random_graphs_weighted_connectivity.pt", # путь к весам модели
            "+model_2.weights=../TNDP_learning/output/inductive_random_graphs_weighted_connectivity.pt", # путь к весам модели
            f"++run_name={initial_routes_name}", # имя запуска, вернет pickle файл с тензором в output_routes
            f"++experiment.cost_function.kwargs.demand_time_weight={demand_time_weight}",
            f"++experiment.cost_function.kwargs.route_time_weight={route_time_weight}",
            f"++experiment.cost_function.kwargs.median_connectivity_weight={median_connectivity_weight}"
        ]
    )

In [5]:
print(OmegaConf.to_yaml(cfg_eval)) # чисто проверка как выглядит конфиг 

experiment:
  logdir: training_logs
  anomaly: false
  cpu: false
  seed: 0
  symmetric_routes: true
  cost_function:
    type: multi
    kwargs:
      mean_stop_time_s: 0
      avg_transfer_wait_time_s: 300
      variable_weights: true
      use_weighted_connectivity: true
      demand_time_weight: 0.33
      route_time_weight: 0.33
      median_connectivity_weight: 0.33
rpc_model:
  common:
    dropout: 0.0
    nonlin_type: ReLU
    embed_dim: 2
  route_generator:
    type: RandomPathCombiningRouteGenerator
    kwargs:
      force_linking_unlinked: false
      halt_prob_is_route_time_weight: true
      logit_clip: null
  backbone_gn:
    net_type: none
    kwargs:
      return_edges: false
      in_node_dim: 4
      in_edge_dim: 14
model_1:
  common:
    dropout: 0.0
    nonlin_type: ReLU
    embed_dim: 64
  route_generator:
    kwargs:
      force_linking_unlinked: false
      logit_clip: null
      n_nodepair_layers: 3
      n_pathscorer_layers: 3
      pathscorer_hidden_dim: 16
  

In [ ]:
pareto_networks, pareto_costs, pareto_metrics = main_ngsaii(cfg_eval)

  2%|▏         | 39/2000 [00:51<46:31,  1.42s/it]

In [7]:
pareto_costs

array([[  855.5684  ,  7680.      ,    96.06363 ],
       [  638.42004 , 17520.      ,    80.04606 ],
       [  624.1233  , 19920.      ,    80.04606 ],
       [  827.0906  ,  8280.      ,    98.72819 ],
       [  855.5684  ,  7680.      ,    96.06363 ],
       [  698.11176 , 10200.      ,    87.871735],
       [  754.79767 ,  9000.      ,    92.080826],
       [  651.1368  , 15120.      ,    84.78809 ],
       [  752.447   , 12000.      ,    81.4291  ],
       [  662.9672  , 14280.      ,    85.002426],
       [  721.73413 , 10560.      ,    84.78809 ],
       [  793.4104  ,  8520.      ,    90.68234 ],
       [  788.55493 ,  8520.      ,    94.989395],
       [  844.85547 ,  7800.      ,    98.40605 ],
       [  749.5568  ,  9960.      ,    91.7894  ],
       [  667.0135  , 12720.      ,    80.13697 ],
       [  636.6859  , 18000.      ,    80.13697 ],
       [  691.09827 , 11280.      ,    89.6081  ],
       [  648.20807 , 15600.      ,    80.42728 ],
       [  641.96533 , 16800.   

In [8]:
pareto_metrics

{'cost': tensor([0.3719, 0.6130, 0.6512, 0.3719, 0.4368, 0.5029, 0.3965, 0.6016, 0.4953,
         0.5474, 0.3965, 0.3965, 0.3965, 0.3719, 0.3965, 0.5029, 0.6362, 0.4501,
         0.5510, 0.6130]),
 'ATT': tensor([14.2595, 10.6320, 10.4509, 14.2595, 13.8304, 12.7694, 13.0629, 10.7425,
         12.0938, 10.9859, 13.0629, 13.0629, 13.0629, 14.2595, 13.0629, 12.7694,
         10.6179, 11.5183, 10.7688, 10.6320]),
 'RTT': tensor([128., 296., 320., 128., 170., 216., 150., 288., 216., 254., 150., 150.,
         150., 128., 150., 216., 310., 192., 258., 296.]),
 '$d_0$': tensor([56.8401, 89.2100, 92.8067, 56.8401, 59.7303, 74.5023, 61.0148, 91.3295,
         69.9422, 84.7784, 61.0148, 61.0148, 61.0148, 56.8401, 61.0148, 74.5023,
         89.8523, 78.5485, 86.7052, 89.2100]),
 '$d_1$': tensor([26.8465, 10.4689,  6.8722, 26.8465, 24.7270, 24.4059, 35.0032,  8.1567,
         29.2229, 14.3866, 35.0032, 35.0032, 35.0032, 26.8465, 35.0032, 24.4059,
          9.6339, 20.1670, 12.6525, 10.4689]),
 '$d

In [ ]:
import csv
from pathlib import Path
from hydra import initialize_config_dir, compose
import torch
from tqdm import tqdm
# Assume main_ngsaii is imported from the appropriate module
# from learning.<appropriate_file> import main_ngsaii

# Путь к конфигам
cfg_dir = Path("../TNDP_learning/cfg").resolve()
model_weights_path = Path("../TNDP_learning/output/inductive_random_graphs_weighted_connectivity.pt").resolve()

datasets = ["mandl", "mumford0", "mumford1", "mumford2"]

weights_combinations = [
    (1, 0, 0),
    (0, 1, 0),
    (0, 0, 1),
    (0.5, 0.5, 0),
    (0.5, 0, 0.5),
    (0, 0.5, 0.5),
    (0.33, 0.33, 0.33),
]

# CSV файл
results_file = Path("eval_nsgaii_datasets2.csv")
if not results_file.exists():
    with open(results_file, mode='w', newline='') as f:
        csv.writer(f).writerow([
            "dataset", "demand_time", "route_time", "connectivity",
            "ATT", "RTT", "median_connectivity", "median_connectivity_weighted", "cost", "$d_{un}$", "$d_0$", "$d_1$", "$d_2$"
        ])

# Запуск экспериментов
for dataset_name in datasets:
    try:
        # Запуск модели один раз с весами 0.33, 0.33, 0.33
        experiment_name = f'exp_weighted_connectivity_{dataset_name}_pp_0.33_op_0.33_cp_0.33'
        initial_routes_name = experiment_name + '_starting'
        generated_routes_name = experiment_name + '_generated'

        with initialize_config_dir(config_dir=str(cfg_dir), version_base=None):
            cfg_eval = compose(
                config_name="nsgaii",
                overrides=[
                    f"+eval={dataset_name}",
                    f"+model_1.weights={str(model_weights_path)}",
                    f"+model_2.weights={str(model_weights_path)}",
                    f"++run_name={initial_routes_name}",
                    f"++experiment.cost_function.kwargs.demand_time_weight=0.33",
                    f"++experiment.cost_function.kwargs.route_time_weight=0.33",
                    f"++experiment.cost_function.kwargs.median_connectivity_weight=0.33"
                ]
            )

        pareto_networks, pareto_costs, pareto_metrics = main_ngsaii(cfg_eval)

        # Перебор всех комбинаций весов для отбора значений из Парето-фронта
        for dt, rt, ct in tqdm(weights_combinations, desc=f"Processing weights for {dataset_name}"):
            try:
                # Вычисление взвешенной суммы (cost) для выбора лучшего решения
                scores = dt * pareto_metrics['ATT'] + rt * pareto_metrics['RTT'] + ct * pareto_metrics['median_connectivity_weighted']
                idx = torch.argmin(scores).item()

                keys_order = ['ATT', 'RTT', 'median_connectivity', "median_connectivity_weighted", 'cost', '$d_{un}$', '$d_0$', '$d_1$', '$d_2$']
                row = [dataset_name, dt, rt, ct] + [
                    round(pareto_metrics[k][idx].item(), 4)
                    for k in keys_order
                ]

                with open(results_file, mode='a', newline='') as f:
                    csv.writer(f).writerow(row)

            except Exception as e:
                print(f"[✗] Failed processing weights {dt}_{rt}_{ct} for {dataset_name}: {e}")

    except Exception as e:
        print(f"[✗] Failed eval for {dataset_name}: {e}")

100%|██████████| 200/200 [00:59<00:00,  3.34it/s]


mutator use counts:
add_terminal: 231.0
delete_terminal: 285.0
add_inside: 114.0
delete_inside: 230.0
invert_nodes: 160.0
exchange_routes: 152.0
replace_node: 181.0
donate_node: 211.0
cost_based_grow: 357.0
cost_based_trim: 301.0
rpc_model_stochastic: 270.0
model_1_stochastic: 225.0
model_1_greedy: 363.0
model_2_stochastic: 256.0
model_2_greedy: 257.0


100%|██████████| 200/200 [01:44<00:00,  1.91it/s]


mutator use counts:
add_terminal: 274.0
delete_terminal: 238.0
add_inside: 177.0
delete_inside: 266.0
invert_nodes: 243.0
exchange_routes: 204.0
replace_node: 180.0
donate_node: 222.0
cost_based_grow: 217.0
cost_based_trim: 323.0
rpc_model_stochastic: 252.0
model_1_stochastic: 276.0
model_1_greedy: 230.0
model_2_stochastic: 238.0
model_2_greedy: 269.0


 54%|█████▎    | 107/200 [03:44<03:08,  2.03s/it]